# Large-Scale Data Analytics Using PySpark – NYC Taxi Trip Analysis

**Big Data Lab Exercise — Full Solution Notebook**

This notebook is designed to run in **Google Colab**. It covers Parts A–G of the lab
(environment setup, data ingestion, exploration, Spark SQL, cleaning & feature engineering,
performance experiments, MLlib regression, and business insights), plus the optional
challenge tasks at the end.

> Run cells top to bottom. Some cells (Part E performance experiment) can take a few minutes.


## Part A — PySpark Environment and Data Ingestion

In [2]:
try:
    import pyspark
    print("PySpark already available:", pyspark.__version__)
except ImportError:
    !pip install -q pyspark
    import pyspark
    print("Installed PySpark:", pyspark.__version__)


PySpark already available: 3.5.1


In [3]:
# 2. Create a SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time

spark = (
    SparkSession.builder
    .appName("NYC_Taxi_BigData_Lab")
    .config("spark.sql.shuffle.partitions", "8")   # small cluster (Colab) -> fewer shuffle partitions
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

spark


In [4]:
# 3. Verify Spark is running and check version
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark version: 3.5.1
Spark master: local[*]
Default parallelism: 2


In [5]:
# 4. Download one month of NYC Yellow Taxi Trip Records (Parquet)
# Official TLC trip record data (public, no auth required)
import urllib.request, os

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
LOCAL_PATH = "yellow_tripdata_2024-01.parquet"

if not os.path.exists(LOCAL_PATH):
    urllib.request.urlretrieve(DATA_URL, LOCAL_PATH)

print("File size (MB):", round(os.path.getsize(LOCAL_PATH) / 1e6, 2))


File size (MB): 49.96


In [6]:
# 5. Load the data into a Spark DataFrame
taxi_df = spark.read.parquet(LOCAL_PATH)
taxi_df.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [7]:
# 6a. Schema (already shown above), 6b. Number of records, 6c. Number of columns
num_records = taxi_df.count()
num_columns = len(taxi_df.columns)
print("Number of records:", num_records)
print("Number of columns:", num_columns)


Number of records: 2964624
Number of columns: 19


In [8]:
# 6d. Sample records
taxi_df.show(5, truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [9]:
# 6e. Basic summary statistics
taxi_df.select(
    "trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count"
).describe().show()


+-------+------------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|   passenger_count|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|           2964624|           2964624|           2964624|           2964624|           2824462|
|   mean|3.6521691789580624|18.175061916792536| 3.335870015894562| 26.80150477092493|1.3392808966805005|
| stddev|225.46257238220082|18.949547705905324|3.8965505998067607|23.385577429672043|0.8502816924800862|
|    min|               0.0|            -899.0|             -80.0|            -900.0|                 0|
|    max|          312722.3|            5000.0|             428.0|            5000.0|                 9|
+-------+------------------+------------------+------------------+------------------+------------------+



In [10]:
# 7. Number of partitions of the DataFrame
print("Number of partitions:", taxi_df.rdd.getNumPartitions())


Number of partitions: 2


### Answers — Part A Questions

**a. Why is Parquet suitable for large-scale analytics?**
Parquet is a columnar, compressed, self-describing binary format. Because it stores data
column-by-column rather than row-by-row, Spark can perform *column pruning* (reading only the
columns a query needs) and *predicate pushdown* (filtering rows before they are even
deserialized), which drastically reduces I/O for analytical (OLAP-style) queries. Its built-in
compression and encoding also reduce storage size and network transfer compared to
row-oriented formats like CSV or JSON, and the embedded schema avoids costly schema inference.

**b. Difference between a Spark DataFrame and a Pandas DataFrame?**
A Pandas DataFrame is a single-machine, in-memory structure — all data must fit in the RAM of
one process, and operations execute eagerly (immediately). A Spark DataFrame is a distributed,
partitioned collection of data spread across the executors of a cluster; operations are
**lazily evaluated** and only executed when an action (e.g. `count()`, `show()`) is called,
after which Spark's Catalyst optimizer builds an execution plan. This lets Spark scale to
datasets far larger than a single machine's memory.

**c. What is a partition in Spark?**
A partition is a logical chunk of a distributed dataset — a subset of rows that resides on one
executor and is processed by one task. Partitions are the fundamental unit of parallelism in
Spark.

**d. Why does Spark divide data into partitions?**
Dividing data into partitions allows Spark to process different chunks of the dataset in
parallel across multiple cores/executors, which is what enables horizontal scalability. It also
allows Spark to schedule work close to where data resides (data locality) and to recover from
failures by recomputing only the lost partitions rather than the whole dataset.


## Part B — Large-Scale Data Exploration

In [11]:
# 1. Total number of taxi trips
total_trips = taxi_df.count()
print("Total taxi trips:", total_trips)


Total taxi trips: 2964624


In [12]:
# 2. Min, max, average trip distance
taxi_df.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),
    F.avg("trip_distance").alias("avg_distance"),
).show()


+------------+------------+------------------+
|min_distance|max_distance|      avg_distance|
+------------+------------+------------------+
|         0.0|    312722.3|3.6521691789580624|
+------------+------------+------------------+



In [13]:
# 3. Min, max, average fare amount
taxi_df.select(
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.avg("fare_amount").alias("avg_fare"),
).show()


+--------+--------+------------------+
|min_fare|max_fare|          avg_fare|
+--------+--------+------------------+
|  -899.0|  5000.0|18.175061916792536|
+--------+--------+------------------+



In [14]:
# 4. Distribution of passenger counts
taxi_df.groupBy("passenger_count").count().orderBy("passenger_count").show()


+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 140162|
|              0|  31465|
|              1|2188739|
|              2| 405103|
|              3|  91262|
|              4|  51974|
|              5|  33506|
|              6|  22353|
|              7|      8|
|              8|     51|
|              9|      1|
+---------------+-------+



In [15]:
# 5. Which hour of day has the highest number of trips?
trips_by_hour = (
    taxi_df
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .count()
    .orderBy(F.desc("count"))
)
trips_by_hour.show(5)
print("Busiest hour:", trips_by_hour.first()["pickup_hour"])


+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|212788|
|         17|206257|
|         16|190201|
|         15|189359|
|         19|184032|
+-----------+------+
only showing top 5 rows

Busiest hour: 18


In [16]:
# 6. Which hour has the highest average fare?
fare_by_hour = (
    taxi_df
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(F.avg("fare_amount").alias("avg_fare"))
    .orderBy(F.desc("avg_fare"))
)
fare_by_hour.show(5)
print("Hour with highest average fare:", fare_by_hour.first()["pickup_hour"])


+-----------+------------------+
|pickup_hour|          avg_fare|
+-----------+------------------+
|          5| 26.61991846088232|
|          4| 22.51864532313935|
|          6|21.650398995872234|
|         23|19.757695974818336|
|          0|19.202658103016713|
+-----------+------------------+
only showing top 5 rows

Hour with highest average fare: 5


In [17]:
# 7. Top 10 pickup locations by number of trips
top10_by_trips = (
    taxi_df.groupBy("PULocationID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)
top10_by_trips.show()


+------------+------+
|PULocationID| count|
+------------+------+
|         132|145240|
|         161|143471|
|         237|142708|
|         236|136465|
|         162|106717|
|         230|106324|
|         186|104523|
|         142|104080|
|         138| 89533|
|         239| 88474|
+------------+------+



In [18]:
# 8. Top 10 pickup locations by total fare revenue
top10_by_revenue = (
    taxi_df.groupBy("PULocationID")
    .agg(F.sum("fare_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)
top10_by_revenue.show()


+------------+------------------+
|PULocationID|     total_revenue|
+------------+------------------+
|         132|   8627709.2399999|
|         138|3712332.2300000116|
|         161|2181762.5400000024|
|         230|1865191.9300000058|
|         237|1737640.0000000065|
|         236|1735009.9600000114|
|         186|1650058.0799999943|
|         162| 1577909.160000016|
|         142| 1397529.979999999|
|         163|1286461.4699999993|
+------------+------------------+



### Answers — Part B Questions

**a. Which operations are transformations and which are actions?**
`select`, `withColumn`, `groupBy`, `agg`, `orderBy`, `filter`, and `limit` are all
**transformations** — they are lazy and just build up a logical plan. `count()`, `show()`,
`collect()`, and `first()` are **actions** — they trigger the actual distributed execution of
the accumulated transformations.

**b. Which operations are likely to cause a shuffle?**
`groupBy(...).agg(...)`, `orderBy`/`sort`, and `distinct()` typically cause a shuffle, because
rows with the same key (e.g. the same `PULocationID` or `pickup_hour`) may live on different
partitions and must be redistributed across the cluster so they can be aggregated together.
`select` and `filter` (row-level, no cross-row dependency) do not require a shuffle.

**c. Why might groupBy operations become expensive for very large datasets?**
`groupBy` requires shuffling data across the network so that all rows sharing a key land on the
same executor — this involves serialization, disk spill, and network I/O, all of which scale
with data volume. If keys are skewed (e.g. one pickup location dominates trip counts), a few
tasks end up handling disproportionately large partitions, creating stragglers that slow down
the whole stage.


## Part C — Spark SQL

In [19]:
# Register the taxi DataFrame as a temporary SQL view
taxi_df.createOrReplaceTempView("taxi")


In [20]:
# 1. Average fare and average trip distance for each passenger count
spark.sql('''
    SELECT passenger_count,
           AVG(fare_amount)   AS avg_fare,
           AVG(trip_distance) AS avg_distance
    FROM taxi
    GROUP BY passenger_count
    ORDER BY passenger_count
''').show()


+---------------+------------------+------------------+
|passenger_count|          avg_fare|      avg_distance|
+---------------+------------------+------------------+
|           NULL|20.016193904200065|11.674403475977822|
|              0|17.075336405529896|2.7438750993166874|
|              1|17.557051804717617|3.1375658449909207|
|              2|20.171285105269323| 3.782764037787955|
|              3|20.041298568955362| 3.664591615349205|
|              4| 21.83379901489209|  3.87591141724711|
|              5|17.511869814361543|3.0734722139318427|
|              6|  17.2285693195544|2.9516888113452344|
|              7|45.411249999999995|           2.29375|
|              8| 81.39098039215685|1.5539215686274512|
|              9|              11.4|               1.8|
+---------------+------------------+------------------+



In [21]:
# 2. Top 10 pickup locations by number of trips (Spark SQL)
spark.sql('''
    SELECT PULocationID, COUNT(*) AS trip_count
    FROM taxi
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
''').show()


+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         132|    145240|
|         161|    143471|
|         237|    142708|
|         236|    136465|
|         162|    106717|
|         230|    106324|
|         186|    104523|
|         142|    104080|
|         138|     89533|
|         239|     88474|
+------------+----------+



In [22]:
# 3. Busiest hours of the day
spark.sql('''
    SELECT HOUR(tpep_pickup_datetime) AS pickup_hour, COUNT(*) AS trip_count
    FROM taxi
    GROUP BY pickup_hour
    ORDER BY trip_count DESC
''').show()


+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         18|    212788|
|         17|    206257|
|         16|    190201|
|         15|    189359|
|         19|    184032|
|         14|    182898|
|         13|    169903|
|         12|    164559|
|         21|    160888|
|         20|    159989|
|         11|    150542|
|         22|    143261|
|         10|    138778|
|          9|    128970|
|          8|    117209|
|         23|    109287|
|          7|     83719|
|          0|     79094|
|          1|     53627|
|          6|     41429|
+-----------+----------+
only showing top 20 rows



In [23]:
# 4. Average fare for trips grouped by hour
spark.sql('''
    SELECT HOUR(tpep_pickup_datetime) AS pickup_hour, AVG(fare_amount) AS avg_fare
    FROM taxi
    GROUP BY pickup_hour
    ORDER BY pickup_hour
''').show(24)


+-----------+------------------+
|pickup_hour|          avg_fare|
+-----------+------------------+
|          0|19.202658103016713|
|          1| 17.52729576519277|
|          2| 16.48288216008736|
|          3| 18.15013461770988|
|          4| 22.51864532313935|
|          5| 26.61991846088232|
|          6|21.650398995872234|
|          7| 18.53918011443055|
|          8| 17.65468274620556|
|          9|17.708415367915187|
|         10| 17.75339376558262|
|         11|17.432059425276748|
|         12|17.476313419503285|
|         13|18.116046450033263|
|         14| 18.94578672265435|
|         15| 18.77463442455851|
|         16|19.121762240997988|
|         17|17.838370576513736|
|         18| 16.72315088256837|
|         19| 17.29479394887858|
|         20|17.695375307052274|
|         21|17.954276204564554|
|         22|18.688929087469532|
|         23|19.757695974818336|
+-----------+------------------+



In [24]:
# 5. Pickup locations with unusually high average fares (> overall mean + 2*stddev)
stats = spark.sql("SELECT AVG(fare_amount) AS m, STDDEV(fare_amount) AS s FROM taxi").first()
threshold = stats["m"] + 2 * stats["s"]
print("Threshold for 'unusually high' avg fare:", round(threshold, 2))

spark.sql(f'''
    SELECT PULocationID, AVG(fare_amount) AS avg_fare, COUNT(*) AS trip_count
    FROM taxi
    GROUP BY PULocationID
    HAVING AVG(fare_amount) > {threshold}
    ORDER BY avg_fare DESC
''').show()


Threshold for 'unusually high' avg fare: 56.07
+------------+-----------------+----------+
|PULocationID|         avg_fare|trip_count|
+------------+-----------------+----------+
|          44|            264.1|         1|
|         187|          102.425|         2|
|         118|            99.37|         9|
|         111|             89.0|         1|
|           1|88.17711864406778|       295|
|         265|82.33112786489747|      1658|
|         109|            72.01|         1|
|         172|             70.0|         2|
|           8|67.67272727272727|        11|
|         204|            62.33|         1|
|         132| 59.4031206279255|    145240|
+------------+-----------------+----------+



In [25]:
# Compare DataFrame API vs Spark SQL for two problems (top pickup locations, busiest hour)
t0 = time.time()
df_api_result = taxi_df.groupBy("PULocationID").count().orderBy(F.desc("count")).limit(10).collect()
t1 = time.time()
sql_result = spark.sql('''
    SELECT PULocationID, COUNT(*) AS trip_count FROM taxi
    GROUP BY PULocationID ORDER BY trip_count DESC LIMIT 10
''').collect()
t2 = time.time()

print(f"DataFrame API time: {t1 - t0:.3f}s")
print(f"Spark SQL time:     {t2 - t1:.3f}s")
print("Results match:", [r['PULocationID'] for r in df_api_result] == [r['PULocationID'] for r in sql_result])


DataFrame API time: 0.373s
Spark SQL time:     0.289s
Results match: True


**Comparison note:** The DataFrame API and Spark SQL versions of the same query (e.g. top
pickup locations, or busiest hour) produce identical results and identical physical plans,
because Spark SQL queries are parsed into the same logical plan representation used by the
DataFrame API and go through the same Catalyst optimizer. Any timing difference between them is
noise (JIT warm-up, caching, scheduling), not a fundamental performance difference.

### Answer — Part C Question

**Why is Spark SQL useful, and how does Spark execute SQL queries?**
Spark SQL lets analysts express complex aggregations declaratively, using familiar SQL syntax,
without hand-writing distributed transformation chains. This is especially useful for
data analysts who know SQL but not the Spark DataFrame API, and for reusing existing SQL-based
tooling and BI dashboards. Under the hood, a SQL query is parsed into an unresolved logical
plan, resolved against the Catalog (schema information) into a logical plan, optimized by the
**Catalyst optimizer** (predicate pushdown, constant folding, column pruning, etc.), converted
into one or more physical plans, and the cheapest physical plan (chosen via cost-based
optimization) is compiled to RDD/Tungsten bytecode and executed across the cluster — the same
engine and optimizer used by the DataFrame API.


## Part D — Data Cleaning and Feature Engineering

In [26]:
# 1 & 2. Identify suspicious records and define filtering rules
suspicious_counts = {
    "zero_or_negative_distance": taxi_df.filter(F.col("trip_distance") <= 0).count(),
    "zero_or_negative_fare":     taxi_df.filter(F.col("fare_amount") <= 0).count(),
    "invalid_passenger_count":   taxi_df.filter((F.col("passenger_count") <= 0) | (F.col("passenger_count") > 6)).count(),
    "extremely_long_distance":   taxi_df.filter(F.col("trip_distance") > 100).count(),
    "extremely_large_fare":      taxi_df.filter(F.col("fare_amount") > 500).count(),
    "invalid_timestamps":        taxi_df.filter(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")).count(),
}
for k, v in suspicious_counts.items():
    print(f"{k}: {v}")


zero_or_negative_distance: 60371
zero_or_negative_fare: 38341
invalid_passenger_count: 31525
extremely_long_distance: 59
extremely_large_fare: 46
invalid_timestamps: 870


In [27]:
# 3. Create a cleaned DataFrame applying reasonable filtering rules
cleaned_df = taxi_df.filter(
    (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100) &
    (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500) &
    (F.col("passenger_count") > 0) & (F.col("passenger_count") <= 6) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

cleaned_count = cleaned_df.count()


In [28]:
# 4. Report how many records were removed
removed = num_records - cleaned_count
print(f"Original records: {num_records}")
print(f"Cleaned records:  {cleaned_count}")
print(f"Removed records:  {removed} ({removed / num_records:.2%})")


Original records: 2964624
Cleaned records:  2723653
Removed records:  240971 (8.13%)


In [29]:
# 5. Derived features: trip duration, pickup hour, day of week, weekend indicator,
#    fare per mile, average speed
cleaned_df = (
    cleaned_df
    .withColumn(
        "trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "E"))  # Mon, Tue, ...
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("tpep_pickup_datetime").isin([1, 7]), 1).otherwise(0)  # 1=Sun,7=Sat
    )
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn(
        "avg_speed_mph",
        F.when(F.col("trip_duration_min") > 0,
               F.col("trip_distance") / (F.col("trip_duration_min") / 60.0))
    )
)

# Additionally remove rows where duration became invalid after feature engineering
cleaned_df = cleaned_df.filter((F.col("trip_duration_min") > 0) & (F.col("trip_duration_min") < 240))

cleaned_df.select(
    "trip_distance", "fare_amount", "trip_duration_min", "pickup_hour",
    "day_of_week", "is_weekend", "fare_per_mile", "avg_speed_mph"
).show(5)


+-------------+-----------+------------------+-----------+-----------+----------+------------------+------------------+
|trip_distance|fare_amount| trip_duration_min|pickup_hour|day_of_week|is_weekend|     fare_per_mile|     avg_speed_mph|
+-------------+-----------+------------------+-----------+-----------+----------+------------------+------------------+
|         1.72|       17.7|              19.8|          0|        Mon|         0|10.290697674418604| 5.212121212121212|
|          1.8|       10.0|               6.6|          0|        Mon|         0| 5.555555555555555|16.363636363636363|
|          4.7|       23.3|17.916666666666668|          0|        Mon|         0| 4.957446808510638|15.739534883720932|
|          1.4|       10.0|               8.3|          0|        Mon|         0| 7.142857142857143|10.120481927710843|
|          0.8|        7.9|               6.1|          0|        Mon|         0|             9.875| 7.868852459016395|
+-------------+-----------+-------------

### Answers — Part D Questions

**a. Why is data cleaning particularly important in Big Data analytics?**
At large scale, even a small percentage of bad records translates into a large absolute number
of corrupted rows, and errors propagate through every downstream aggregation, model, and
dashboard. Because Big Data pipelines are often automated and re-run regularly, uncaught data
quality issues silently bias results over and over rather than being caught once by a human
reviewing a small spreadsheet.

**b. What effect could extreme outliers have on your analysis?**
Outliers (e.g. a $50,000 fare or a trip logged as lasting 10,000 minutes) can dominate
sum/average-based aggregations, distort correlation and regression coefficients, inflate error
metrics for ML models, and mislead business conclusions (e.g. making a low-traffic zone look
like a high-revenue zone because of one erroneous fare).

**c. Which features appear to be most useful for understanding taxi demand and revenue?**
`pickup_hour` and `day_of_week`/`is_weekend` are most useful for understanding **demand**
patterns (when trips happen), while `PULocationID`, `trip_distance`, and `fare_per_mile` are
most useful for understanding **revenue** patterns (where money is made and how efficiently).
`avg_speed_mph` is a useful proxy for traffic/operational efficiency.


## Part E — Spark Performance Experiment

Aggregation chosen: **average fare by pickup location** (`PULocationID`).


In [30]:
def run_aggregation(df):
    return df.groupBy("PULocationID").agg(F.avg("fare_amount").alias("avg_fare")).collect()

results = []


In [31]:
# Experiment 1: original DataFrame
print("Original partitions:", cleaned_df.rdd.getNumPartitions())

t0 = time.time()
run_aggregation(cleaned_df)
t1 = time.time()
exp1_time = t1 - t0
results.append(("Original", cleaned_df.rdd.getNumPartitions(), "No", round(exp1_time, 3)))
print(f"Experiment 1 (Original) time: {exp1_time:.3f}s")


Original partitions: 2
Experiment 1 (Original) time: 0.910s


In [32]:
# Experiment 2: repartitioned DataFrame
repartitioned_df = cleaned_df.repartition(16)
print("Repartitioned partitions:", repartitioned_df.rdd.getNumPartitions())

t0 = time.time()
run_aggregation(repartitioned_df)
t1 = time.time()
exp2_time = t1 - t0
results.append(("Repartitioned", repartitioned_df.rdd.getNumPartitions(), "No", round(exp2_time, 3)))
print(f"Experiment 2 (Repartitioned) time: {exp2_time:.3f}s")


Repartitioned partitions: 16
Experiment 2 (Repartitioned) time: 4.958s


In [33]:
# Experiment 3: cache/persist and repeat
cached_df = cleaned_df.cache()
cached_df.count()  # materialize the cache

t0 = time.time()
run_aggregation(cached_df)
t1 = time.time()
exp3_time_first = t1 - t0

t0 = time.time()
run_aggregation(cached_df)
t1 = time.time()
exp3_time_second = t1 - t0

results.append(("Cached (1st run)", cached_df.rdd.getNumPartitions(), "Yes", round(exp3_time_first, 3)))
results.append(("Cached (2nd run)", cached_df.rdd.getNumPartitions(), "Yes", round(exp3_time_second, 3)))

print(f"Experiment 3 (Cached, 1st run):  {exp3_time_first:.3f}s")
print(f"Experiment 3 (Cached, 2nd run):  {exp3_time_second:.3f}s")


Experiment 3 (Cached, 1st run):  0.556s
Experiment 3 (Cached, 2nd run):  0.284s


In [34]:
# Results table
import pandas as pd  # display only — not used for the actual Spark computation
results_df = pd.DataFrame(results, columns=["Experiment", "Number of Partitions", "Cached?", "Execution Time (s)"])
results_df


,Experiment,Number of Partitions,Cached?,Execution Time (s)
0,Original,2,No,0.910
1,Repartitioned,16,No,4.958
2,Cached (1st run),2,Yes,0.556
3,Cached (2nd run),2,Yes,0.284


> Note: `pandas` is used here **only** to pretty-print the small timing-results table for
> the report — all Big Data processing above was performed with PySpark, per the lab's
> requirement not to use Pandas/Scikit-learn for the main analysis.

### Answers — Part E Questions

**a. Did increasing the number of partitions always improve performance?**
Not necessarily — record actual timings above. In a small Colab environment (few cores),
increasing partitions beyond the number of available cores adds *scheduling and task-launch
overhead* without adding real parallelism, so performance can plateau or worsen.

**b. Why can too many partitions also reduce performance?**
Each partition/task has fixed overhead (task scheduling, serialization, small file/shuffle
block bookkeeping). If partitions are too small, this per-task overhead dominates the actual
compute time. Excessive partitions can also produce many tiny shuffle files, adding I/O
overhead.

**c. What is the purpose of `cache()` or `persist()`?**
They store a DataFrame's computed result (in memory and/or on disk, depending on the storage
level) so that subsequent actions reuse it instead of recomputing the full lineage from the
original data source. This is valuable when the same DataFrame is reused across multiple
actions or ML iterations.

**d. Why can the first execution after caching behave differently from subsequent executions?**
The first action after calling `cache()` still has to *compute and materialize* the DataFrame
(reading the source, applying transformations) *and* write the result into cache — so it is not
faster than an uncached run. Only from the second action onward does Spark read the
already-materialized data directly from memory/disk cache, skipping recomputation, which is why
timings usually drop noticeably on the second run.

**e. What is meant by data shuffling in Spark?**
Shuffling is the process of redistributing data across partitions/executors over the network so
that rows with the same key end up together — required for operations like `groupBy`, `join`,
and `distinct`. It involves writing intermediate data to disk, transferring it across the
network, and re-reading it, making it one of the most expensive operations in Spark.


## Part F — Large-Scale Machine Learning with Spark MLlib

In [35]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline


In [36]:
# 1 & 2. Select features and prepare the feature vector
ml_df = (
    cleaned_df
    .withColumn("day_of_week_idx_input", F.col("day_of_week"))
    .select(
        "trip_distance", "passenger_count", "trip_duration_min",
        "pickup_hour", "day_of_week_idx_input", "fare_amount"
    )
    .na.drop()
)

day_indexer = StringIndexer(inputCol="day_of_week_idx_input", outputCol="day_of_week_idx")

feature_cols = ["trip_distance", "passenger_count", "trip_duration_min", "pickup_hour", "day_of_week_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")


In [37]:
# 3. Train/test split
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())


Training rows: 2177617
Test rows: 544197


In [38]:
# 4 & 5. Choose an algorithm and train the model
# RandomForestRegressor is chosen because it: (1) handles non-linear relationships between
# distance/duration/hour and fare, (2) is robust to outliers/skewed features without needing
# feature scaling, and (3) scales well as a distributed Spark ML algorithm.
rf = RandomForestRegressor(featuresCol="features", labelCol="fare_amount", numTrees=50, maxDepth=8, seed=42)

pipeline = Pipeline(stages=[day_indexer, assembler, rf])

model = pipeline.fit(train_df)


In [39]:
# 6. Generate predictions on the test set
predictions = model.transform(test_df)
predictions.select("trip_distance", "trip_duration_min", "pickup_hour", "fare_amount", "prediction").show(10)


+-------------+--------------------+-----------+-----------+-----------------+
|trip_distance|   trip_duration_min|pickup_hour|fare_amount|       prediction|
+-------------+--------------------+-----------+-----------+-----------------+
|         0.01|0.016666666666666666|         14|       50.0|7.613219563581207|
|         0.01| 0.03333333333333333|         11|       87.0|7.555963769554104|
|         0.01| 0.03333333333333333|         11|       82.0|7.530043057987971|
|         0.01| 0.03333333333333333|         18|       87.3|7.613219563581207|
|         0.01|                0.05|          7|       82.0|7.723491811652589|
|         0.01|                0.05|         14|        3.0|7.490754091435775|
|         0.01|                0.05|         20|       90.0|7.530167006541664|
|         0.01| 0.06666666666666667|          4|       82.0|8.072128894020478|
|         0.01| 0.06666666666666667|         14|       82.0|7.471007088358342|
|         0.01| 0.06666666666666667|         15|    

In [40]:
# 7. Evaluate the model
evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
mae  = evaluator_mae.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)

print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R2:   {r2:.3f}")


RMSE: 6.196
MAE:  1.635
R2:   0.871


In [41]:
# Feature importances
rf_model = model.stages[-1]
for name, importance in zip(feature_cols, rf_model.featureImportances.toArray()):
    print(f"{name}: {importance:.4f}")


trip_distance: 0.6527
passenger_count: 0.0023
trip_duration_min: 0.3406
pickup_hour: 0.0029
day_of_week_idx: 0.0016


### Answers — Part F Questions

**a. Why is this problem treated as regression rather than classification?**
The target variable, `fare_amount`, is a continuous numeric value (dollars, potentially any
real number within a range), not a discrete set of categories/labels — so the task is to
predict a quantity, which is the definition of a regression problem.

**b. Which features appear to contribute most to the prediction?** *(see feature importances
printed above — in practice `trip_distance` and `trip_duration_min` are typically the strongest
predictors, since NYC taxi fares are largely metered by distance and time.)*

**c. What are the limitations of your model?**
The model does not include pickup/dropoff location, traffic conditions, tolls, surcharges, or
tip amount, all of which affect the true `total_amount`. It is trained on a single month of
data and may not generalize to seasonal effects, fare-rule changes, or other boroughs/years. A
tree ensemble can also struggle to extrapolate to rare, very long or very expensive trips that
were filtered out or under-represented in training.

**d. Would you expect the model to perform equally well for all types of taxi trips? Explain.**
No. Short, common trips in dense areas are well represented in the training data, so predictions
there tend to be more accurate. Very long trips (e.g. airport runs), trips with unusual
duration-to-distance ratios (heavy traffic), or trips in undersampled locations/hours are likely
to have higher prediction error, because the model has seen fewer comparable examples and those
trips often follow different fare structures (flat rates, surcharges).


## Part G — Business Insights

Based on the analysis above, here are actionable, evidence-backed recommendations. Replace the
bracketed placeholders with the actual numbers/IDs your run produced (hour, `PULocationID`
values, etc.) before submitting.

1. **Align driver supply with peak demand hours.** The hourly trip-count analysis (Part B, Q5)
   identified the busiest pickup hour. The taxi company should incentivize more drivers to be
   active (e.g. through surge-style bonuses or shift scheduling) during this window to reduce
   passenger wait times and capture demand that would otherwise be lost to competitors.

2. **Reposition idle vehicles toward high-revenue zones.** The top-10 pickup locations by total
   fare revenue (Part B, Q8 / Part C, Q2) are disproportionately valuable compared to
   high-*volume* but lower-fare zones. Directing idle drivers toward these zones — rather than
   simply the busiest zones by trip count — can increase revenue per driver-hour.

3. **Investigate locations with unusually high average fares (Part C, Q5).** These zones may
   indicate legitimate long-distance/airport routes worth actively serving, or they may signal
   data-quality/fraud issues (e.g. mis-metered fares) worth auditing — either way, the pattern
   is actionable.

4. **Adjust weekday vs weekend fleet strategy.** The `is_weekend` feature engineered in Part D
   can be used to compare weekday commuter demand against weekend leisure demand; staffing and
   pricing strategy can be tuned separately for each pattern rather than using a single
   citywide plan.

5. **Use the fare-prediction model for dynamic pricing sanity checks.** The MLlib regression
   model (Part F) predicts an expected fare from trip characteristics; large deviations between
   predicted and actual fares can flag anomalous trips for review, and the model's feature
   importances highlight which trip factors most influence pricing, informing rate-card design.


## Challenge Tasks (Optional)

### Challenge 1 — Scaling the Dataset
Combine multiple months of NYC taxi data and compare execution time as data size grows.


In [42]:
# Example: download a second month and union it to observe how execution time scales
DATA_URL_2 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet"
LOCAL_PATH_2 = "yellow_tripdata_2024-02.parquet"

if not os.path.exists(LOCAL_PATH_2):
    urllib.request.urlretrieve(DATA_URL_2, LOCAL_PATH_2)

taxi_df_2 = spark.read.parquet(LOCAL_PATH_2)
combined_df = taxi_df.unionByName(taxi_df_2)

t0 = time.time()
combined_count = combined_df.count()
t1 = time.time()

print(f"Combined record count: {combined_count} (vs {num_records} for one month)")
print(f"Count() execution time on combined data: {t1 - t0:.3f}s")


Combined record count: 5972150 (vs 2964624 for one month)
Count() execution time on combined data: 0.354s


### Challenge 2 — Partition Investigation
Sweep several partition counts on the same aggregation and compare timings to find an
approximate sweet spot for this Colab environment.


In [43]:
partition_options = [1, 4, 8, 16, 32, 64]
partition_results = []

for n in partition_options:
    test_df_part = cleaned_df.repartition(n)
    t0 = time.time()
    run_aggregation(test_df_part)
    t1 = time.time()
    partition_results.append((n, round(t1 - t0, 3)))
    print(f"Partitions={n}: {t1 - t0:.3f}s")

pd.DataFrame(partition_results, columns=["Partitions", "Execution Time (s)"])


Partitions=1: 1.066s
Partitions=4: 3.054s
Partitions=8: 3.051s
Partitions=16: 2.556s
Partitions=32: 2.959s
Partitions=64: 3.500s


,Partitions,Execution Time (s)
0,1,1.066
1,4,3.054
2,8,3.051
3,16,2.556
4,32,2.959
5,64,3.500


*Discussion:* the optimal partition count generally tracks the number of available CPU
cores in the Colab runtime (`spark.sparkContext.defaultParallelism`) — too few partitions
under-utilizes cores, while too many adds scheduling overhead, so the best setting is
environment-dependent rather than a fixed universal number.

### Challenge 3 — Alternative ML Algorithm
Train a Gradient-Boosted Tree regressor as a second model and compare against the Random Forest.


In [44]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount", maxIter=50, maxDepth=6, seed=42)
gbt_pipeline = Pipeline(stages=[day_indexer, assembler, gbt])
gbt_model = gbt_pipeline.fit(train_df)

gbt_predictions = gbt_model.transform(test_df)

gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_mae  = evaluator_mae.evaluate(gbt_predictions)
gbt_r2   = evaluator_r2.evaluate(gbt_predictions)

print("Random Forest -> RMSE: {:.3f}, MAE: {:.3f}, R2: {:.3f}".format(rmse, mae, r2))
print("GBT           -> RMSE: {:.3f}, MAE: {:.3f}, R2: {:.3f}".format(gbt_rmse, gbt_mae, gbt_r2))


Random Forest -> RMSE: 6.196, MAE: 1.635, R2: 0.871
GBT           -> RMSE: 6.122, MAE: 1.558, R2: 0.874


*Trade-offs:* Gradient-Boosted Trees often achieve slightly lower error than Random
Forests because boosting sequentially corrects prior errors, but they train slower (sequential,
not embarrassingly parallel across trees) and are more prone to overfitting if `maxIter`/
`maxDepth` are not tuned carefully. Random Forest trains trees independently (easier to
parallelize) and is generally more robust to overfitting.

### Challenge 4 — Demand Prediction
Reformulate the problem to predict **trip demand (count) per pickup-location-hour** instead of
per-trip fare.


In [45]:
# Build an hourly-demand-per-location training table
demand_df = (
    cleaned_df
    .groupBy("PULocationID", "pickup_hour", "day_of_week", "is_weekend")
    .agg(F.count("*").alias("trip_count"))
)

demand_indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_idx")
demand_assembler = VectorAssembler(
    inputCols=["PULocationID", "pickup_hour", "day_of_week_idx", "is_weekend"],
    outputCol="features"
)

demand_train, demand_test = demand_df.randomSplit([0.8, 0.2], seed=42)

demand_rf = RandomForestRegressor(featuresCol="features", labelCol="trip_count", numTrees=50, seed=42)
demand_pipeline = Pipeline(stages=[demand_indexer, demand_assembler, demand_rf])
demand_model = demand_pipeline.fit(demand_train)

demand_predictions = demand_model.transform(demand_test)

demand_evaluator = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="rmse")
print("Demand model RMSE:", demand_evaluator.evaluate(demand_predictions))


Demand model RMSE: 226.96412593019957


**Target variable:** `trip_count` — number of trips originating from a given
`PULocationID` in a given `pickup_hour`/day-of-week combination.
**Features:** `PULocationID`, `pickup_hour`, `day_of_week` (indexed), `is_weekend`.
This reformulates the original fare-prediction problem into an **operational demand-forecasting**
problem, which is arguably more directly useful for fleet positioning than fare prediction
alone.


## Conclusion

This notebook used PySpark end-to-end — DataFrame API, Spark SQL, MLlib — to analyze one month
of NYC Yellow Taxi trip records without relying on Pandas or Scikit-learn for the core
processing. Key takeaways:

- The dataset shows clear temporal demand patterns (peak pickup hours) and spatial revenue
  concentration (a small number of pickup zones account for a disproportionate share of
  revenue).
- Real-world data required non-trivial cleaning (invalid distances, fares, passenger counts,
  and timestamps), underscoring why data-quality steps are essential before any downstream
  analytics or modeling.
- Partitioning and caching materially affect execution time, and the "best" configuration is
  resource-dependent rather than fixed — this was demonstrated empirically in Part E and
  Challenge 2.
- A Random Forest regression model built with Spark MLlib can predict fare amount from trip
  characteristics with reasonable accuracy, with trip distance and duration as the dominant
  predictors, though it has known limitations (no location/traffic/surcharge features).
- These findings translate into concrete operational recommendations for the taxi company
  around driver scheduling, vehicle repositioning, and anomaly detection.

**Remember before submitting:** run every cell end-to-end in a fresh Colab runtime, fill in the
actual numeric results into Part E's timing table and Part G's bracketed placeholders, and add
your own commentary/graphs where you'd like to expand on the auto-generated tables above.
